# Access Policies and Applications

```{contents}
:local:
:depth: 2
```


```{index} stack; queue; deque; choosing a data structure
```



```{index} Choosing the Access Rule
```

## Choosing the Access Rule



Stacks, queues, and deques are easiest to choose when you name the rule for the next item:

| Situation | Next item should be | Structure |
|---|---|---|
| Undo the most recent edit | Most recently saved | Stack |
| Serve customers fairly | Earliest waiting | Queue |
| Process both urgent and normal work | Either front or back | Deque |

The access rule is more important than the English noun. A "list of tasks" might be a stack, a queue, or a deque depending on how tasks should be removed.


In [ ]:
using System;
using System.Collections.Generic;

var urgent = new Stack<string>();
urgent.Push("Restart failed service");
urgent.Push("Rollback broken update");

var normal = new Queue<string>();
normal.Enqueue("Update documentation");
normal.Enqueue("Clean old logs");

if (urgent.Count > 0)
{
    Console.WriteLine($"Handle now: {urgent.Pop()}");
}
else if (normal.Count > 0)
{
    Console.WriteLine($"Handle next: {normal.Dequeue()}");
}


Handle now: Rollback broken update


This example uses two structures because there are two policies: urgent tasks use most-recent-first, and normal tasks use first-come-first-served.


```{index} backtracking; stack
```



```{index} Backtracking with a Stack
```

## Backtracking with a Stack



Backtracking means trying a path, then returning to an earlier choice if that path does not work. A stack records the path because the most recent choice is the first one to undo.

The pattern has three moves. *Advance* by pushing the choice you are trying. *Fail* by popping it when it leads nowhere, which automatically exposes the previous choice as the new top. *Resume* from that top with the next untried alternative. Nothing earlier in the path is touched, because the stack guarantees that the most recent decision is the first one revisited. The example explores rooms in a building: the path to the current room is exactly the stack's contents, and a dead end is handled by one pop. Chapter 25 turns this pattern into a general search procedure; here the goal is to see why the stack is the right memory for it.

In [ ]:
using System;
using System.Collections.Generic;

var path = new Stack<string>();
path.Push("Start");
path.Push("Hallway");
path.Push("Storage room");

Console.WriteLine($"Dead end at {path.Pop()}");
Console.WriteLine($"Return to {path.Peek()}");
path.Push("Server room");

Console.WriteLine("Current path:");
foreach (string location in path)
{
    Console.WriteLine(location);
}


Dead end at Storage room
Return to Hallway
Current path:
Server room
Hallway
Start


`Stack<T>` enumeration visits the top first, so the displayed path starts with the current location and moves backward through recent choices.


```{index} A Stack Can Replace the Call Stack
```

### A Stack Can Replace the Call Stack

Every method call in C# is itself recorded on a stack. When a method calls another, the runtime pushes the caller's position and local variables; when the callee returns, that record is popped and the caller resumes. A recursive method is therefore a stack-based algorithm whose stack the language manages for you. The connection runs the other way as well: any recursive procedure can be rewritten as a loop over an explicit `Stack<T>` of pending work, which is useful when recursion would go too deep or when the pending items need to be inspected.

The example totals the cost of a product assembled from parts, where each part may itself be assembled from other parts. A recursive version would call itself once per child; the explicit version pushes the children instead and loops until the stack is empty.

In [ ]:
using System;
using System.Collections.Generic;

// Each part lists its direct components; a part with no entry is purchased, not assembled.
var components = new Dictionary<string, string[]>
{
    ["desk"] = new[] { "top", "frame", "drawer" },
    ["frame"] = new[] { "leg", "leg", "leg", "leg", "brace" },
    ["drawer"] = new[] { "box", "handle" },
};
var unitCost = new Dictionary<string, decimal>
{
    ["top"] = 80m, ["leg"] = 12m, ["brace"] = 9m, ["box"] = 25m, ["handle"] = 4m,
};

var pending = new Stack<string>();
pending.Push("desk");
decimal total = 0;
int maxDepth = 0;
while (pending.Count > 0)
{
    maxDepth = Math.Max(maxDepth, pending.Count);
    string part = pending.Pop();
    if (components.TryGetValue(part, out string[]? children))
    {
        foreach (string child in children) pending.Push(child);   // defer the children
    }
    else
    {
        total += unitCost[part];                                     // a purchased part
    }
}
Console.WriteLine($"total cost = {total}");
Console.WriteLine($"most items pending at once = {maxDepth}");

total cost = 166
most items pending at once = 6


The total is 166: a top at 80, four legs at 12, a brace at 9, a box at 25, and a handle at 4. The stack never held more than six items, which is the explicit counterpart of the recursion depth plus the number of siblings waiting at each level. Two details distinguish this from the queue-based worklist in the next subsection. Children are pushed, so the most recently discovered part is processed next and the program finishes one assembly before starting another, the depth-first order. And the order of the result does not depend on that choice: a sum is the same in any order, which is why either structure works here and why the *order* of processing, not the total, is what you choose a structure for.

```{index} breadth-first order; queue
```



```{index} Breadth-First Processing with a Queue
```

## Breadth-First Processing with a Queue



Queues are central to breadth-first processing. When each item can discover more items, newly discovered items go to the back so earlier layers finish first.


In [ ]:
using System;
using System.Collections.Generic;

var pending = new Queue<string>();
pending.Enqueue("home");
var visited = new HashSet<string> { "home" };

var links = new Dictionary<string, string[]>
{
    ["home"] = new[] { "products", "support" },
    ["products"] = new[] { "pricing" },
    ["support"] = new[] { "faq", "contact" },
    ["pricing"] = Array.Empty<string>(),
    ["faq"] = Array.Empty<string>(),
    ["contact"] = Array.Empty<string>()
};

while (pending.Count > 0)
{
    string page = pending.Dequeue();
    Console.WriteLine(page);
    foreach (string next in links[page])
    {
        if (visited.Add(next)) pending.Enqueue(next);
    }
}


home
products
support
pricing
faq
contact


This is the same queue pattern used later for breadth-first graph traversal. The queue preserves discovery order across layers.


```{index} Guided Practice: Visit Pages in Arrival Order
```

### Guided Practice: Visit Pages in Arrival Order


The crawler below uses the wrong collection, so it prints the pages in reverse. Replace it with the collection whose access rule is first-in, first-out so the output is `home`, `products`, `support`.


In [ ]:
using System;
using System.Collections.Generic;

// Your code starts here.
var pending = new Stack<string>();
pending.Push("home"); pending.Push("products"); pending.Push("support");
while (pending.Count > 0) Console.WriteLine(pending.Pop());
// Your code ends here.


In [ ]:
using System;
using System.Collections.Generic;

// Solution
var pending = new Queue<string>();
pending.Enqueue("home"); pending.Enqueue("products"); pending.Enqueue("support");
while (pending.Count > 0) Console.WriteLine(pending.Dequeue());


home
products
support


Only the collection and its three verbs changed; the loop shape is identical. That is the point of this page: the access rule is the design decision, and the structure follows from it.


```{index} stack; operation cost; queue; operation cost
```



```{index} Comparing Costs
```

## Comparing Costs



For the restricted operations, these structures are usually designed to make the main access rule fast:

| Structure | Operation | Typical cost |
|---|---|---:|
| Growable array stack | Push / pop | O(1) amortized push; O(n) growth; O(1) pop |
| Bounded ring queue | Enqueue / dequeue | O(1); full enqueue rejects |
| Growable array queue | Enqueue / dequeue | O(1) amortized enqueue; O(n) growth; O(1) dequeue |
| Deque | Add/remove either end | O(1) with the right representation |
| List used as queue | Remove index 0 | O(n) |

The tempting implementation is not always the right one. Removing from the front of a plain array or list usually shifts many values.

Read the table by asking what each structure had to give up for its fast operations. The array stack gives up nothing but spare capacity because both of its operations happen at the same end, where no element is ever in the way. The queue's two ends are different, so a plain array would have to shift on every dequeue; the ring buys O(1) by letting the front move instead of the data. A list used as a queue pays the shifting cost that the ring avoids, and the example below makes it visible.

In [ ]:
using System;
using System.Collections.Generic;

var slowQueue = new List<string> { "A", "B", "C" };
string first = slowQueue[0];
slowQueue.RemoveAt(0);

Console.WriteLine(first);
Console.WriteLine(string.Join(", ", slowQueue));


A
B, C


`RemoveAt(0)` is simple to read, but it moves the remaining elements left. `Queue<T>` communicates the intended operation and avoids relying on a slow pattern.


```{index} undo and redo require two histories
```



```{index} Undo and Redo Require Two Histories
```

## Undo and Redo Require Two Histories



A realistic undo operation restores a prior state, rather than just printing an action name. Save the previous report state on undo before an edit. Undo saves the current state on redo and restores the latest undo state. Redo reverses that transfer.

| Request | Current state | Undo, bottom to top | Redo, bottom to top |
| --- | --- | --- | --- |
| Start | Draft | Empty | Empty |
| Edit Reviewed | Reviewed | Draft | Empty |
| Edit Approved | Approved | Draft, Reviewed | Empty |
| Undo | Reviewed | Draft | Approved |
| Redo | Approved | Draft, Reviewed | Empty |
| Undo; edit Revised | Revised | Draft, Reviewed | Empty |

A new edit after undo clears redo because it creates a new branch of history. This small editor stores complete immutable string snapshots. Large editors may store inverse commands or smaller changes to limit memory use.

**Predict:** Undo twice after the final edit. What states return? What should an extra undo do when its history is empty?


In [ ]:
using System;
using System.Collections.Generic;
string current = "Draft";
var undo = new Stack<string>();
var redo = new Stack<string>();
void Edit(string next)
{
    undo.Push(current); current = next; redo.Clear();
}
bool Undo()
{
    if (!undo.TryPop(out string? previous)) return false;
    redo.Push(current); current = previous; return true;
}
bool Redo()
{
    if (!redo.TryPop(out string? next)) return false;
    undo.Push(current); current = next; return true;
}
Edit("Reviewed"); Edit("Approved");
Undo(); Console.WriteLine($"after undo={current}");
Redo(); Console.WriteLine($"after redo={current}");
Undo(); Edit("Revised");
Console.WriteLine($"after edit={current}, redo available={Redo()}");


after undo=Reviewed
after redo=Approved
after edit=Revised, redo available=False


```{index} worklist order, duplicate discovery, and fairness
```



```{index} Worklist Order, Duplicate Discovery, and Fairness
```

## Worklist Order, Duplicate Discovery, and Fairness



For the page-link example, after processing home the queue is `[products, support]`. Products adds pricing behind support; support then adds faq and contact. The processing order is home, products, support, pricing, faq, contact. Newly discovered pages join the back, so earlier layers finish first.

Mark a page discovered when it is enqueued. If both products and support link to pricing, only the first discovery enqueues it. Marking only when dequeuing can put duplicate pending entries into the queue. A visited set also prevents a link back to home from causing endless processing. The graph chapters develop the full traversal argument; here the important fact is the worklist policy.

Always serving urgent tickets before normal tickets may starve normal work if urgent arrivals never stop. A deque that adds urgency at the front changes service order; it does not guarantee fairness. A policy such as at most two urgent tickets before one waiting normal ticket makes a different promise.

**Design:** Trace that two-to-one policy for urgent U1, U2, U3 and normal N1, N2. Specify ordering within each group and behavior when either group is empty. What additional policy would you need if tickets have different processing times?


```{index} Code Tracing: Undo and Redo Histories
```

### Code Tracing: Undo and Redo Histories

Using the `Edit`, `Undo`, and `Redo` methods from the previous subsection, trace the sequence below. After each line write `current`, the undo stack bottom to top, and the redo stack bottom to top. Then predict the four printed lines.

```csharp
Edit("A"); Edit("B"); Edit("C");
Undo(); Undo();
Console.WriteLine(current);
Redo();
Console.WriteLine(current);
Edit("D");
Console.WriteLine($"{current} redo={Redo()}");
Undo(); Undo(); Undo();
Console.WriteLine($"{current} undo={Undo()}");
```

In [ ]:
// Solution
using System;
using System.Collections.Generic;

string current = "Draft";
var undo = new Stack<string>();
var redo = new Stack<string>();
void Edit(string next) { undo.Push(current); current = next; redo.Clear(); }
bool Undo() { if (!undo.TryPop(out string? p)) return false; redo.Push(current); current = p; return true; }
bool Redo() { if (!redo.TryPop(out string? n)) return false; undo.Push(current); current = n; return true; }

Edit("A"); Edit("B"); Edit("C");      // current C, undo [Draft, A, B], redo []
Undo(); Undo();                       // current A, undo [Draft], redo [C, B]
Console.WriteLine(current);
Redo();                               // current B, undo [Draft, A], redo [C]
Console.WriteLine(current);
Edit("D");                            // current D, undo [Draft, A, B], redo [] (cleared)
Console.WriteLine($"{current} redo={Redo()}");
Undo(); Undo(); Undo();               // back to Draft; undo [], redo [D, B, A]
Console.WriteLine($"{current} undo={Undo()}");

A
B
D redo=False
Draft undo=False


The lines are `A`, `B`, `D redo=False`, and `Draft undo=False`. The step most often traced wrongly is `Edit("D")`: it pushes `B` onto undo and clears redo, so `C` is gone for good, which is what the `redo=False` confirms. The three undos then walk back through `B`, `A`, and `Draft`, and a fourth returns `false` without changing the state, which is the empty-history policy the method chose.

```{index} validate behavior before measuring
```



```{index} Validate Behavior Before Measuring
```

## Validate Behavior Before Measuring



Test each contract independently: stack LIFO, queue FIFO, deque both ends, and peek without removal. Include empty access, singleton removal and reuse, duplicates, default values, and failure without mutation. For a bounded queue, also test full rejection and wraparound after several cycles.

The same operations can be compared with `Stack<T>` or `Queue<T>` as reference models. Count should agree after each operation; removed values and peek results must agree too. Tests are evidence, while invariant reasoning explains why every legal operation preserves the access rule.

**Transfer:** A service desk needs a queue for waiting tickets and a stack for recently completed tickets that may be restored. Explain why these are two contracts. Describe how to restore a completed ticket and whether it returns to the front or back of the waiting line. State the fairness consequence before choosing a deque.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A crawler marks a page visited when it is dequeued instead of when it is enqueued. The output still lists every page once. What is wrong?

```{dropdown} Answer
The queue can hold the same page several times, once per link that pointed to it before it was processed. The output is deduplicated only because the check happens at dequeue; the wasted queue space and the misleading `Count` are the defect, and on a densely linked site the queue can grow far larger than the number of pages.
```

2. Replacing the queue in the crawler with a stack still visits every reachable page. What changes, and when does it matter?

```{dropdown} Answer
The order changes from layer by layer to deep first along one chain of links. It matters whenever the order is part of the requirement: shortest click distance, fairness between branches, or a progress display that should finish nearby pages first.
```

3. Why does a new edit after an undo clear the redo history rather than keep it?

```{dropdown} Answer
Redo promises to restore the state that the undo removed, which assumed the current state had not changed since. After a new edit the document is on a different branch, and restoring the old redo state would discard the new edit silently. Clearing makes the promise honest.
```

4. The explicit-stack assembly example and a recursive version compute the same total. Give one reason to prefer each.

```{dropdown} Answer
Prefer recursion when the structure is naturally nested and depth is small, because the code is shorter and the language manages the stack. Prefer the explicit stack when depth could exceed the runtime's call-stack limit or when the program needs to inspect, pause, or reorder the pending work, which a hidden call stack does not allow.
```

```{index} stack; queue; exercise
```



```{index} Exercise
```

## Exercise



Two urgent tickets and two normal tickets are waiting. Print one urgent ticket if any exist, otherwise one normal ticket, otherwise `No tickets waiting.` Expected output: `U-2`, because urgent work is handled most-recent-first. Then remove both `Push` calls and predict the new output.


In [ ]:
// Exercise: Split urgent and normal tickets
// Urgent tickets should be handled most-recent-first.
// Normal tickets should be handled first-come-first-served.
// Print one urgent ticket if any exist; otherwise print one normal ticket.

using System;
using System.Collections.Generic;

var urgent = new Stack<string>();
var normal = new Queue<string>();

urgent.Push("U-1");
urgent.Push("U-2");
normal.Enqueue("N-1");
normal.Enqueue("N-2");

// Your code starts here.



// Your code ends here.


In [1]:
// Solution
using System;
using System.Collections.Generic;

var urgent = new Stack<string>();
var normal = new Queue<string>();

urgent.Push("U-1");
urgent.Push("U-2");
normal.Enqueue("N-1");
normal.Enqueue("N-2");

if (urgent.Count > 0)
{
    Console.WriteLine(urgent.Pop());
}
else if (normal.Count > 0)
{
    Console.WriteLine(normal.Dequeue());
}
else
{
    Console.WriteLine("No tickets waiting.");
}


U-2


```{index} summary
```



```{index} Summary
```

## Summary



Stacks, queues, and deques are small ADTs with big design value. Once the access rule is clear, the implementation and the C# library type usually become easier to choose.


```{rubric} Footnotes
```


[^1503_applications_lab-1]: Breadth-first graph traversal is covered in more depth in later graph chapters; here the goal is to see why a queue fits the processing rule.
